# IMDB 影评二分类 - 使用早停与模型保存优化训练

本实验在基础二分类模型上，加入两种重要的训练优化机制：

## 1. EarlyStopping（早停）
当模型在验证集上的表现不再变好时，自动停止训练。
- 防止继续无意义训练
- 减少过拟合
- 节省训练时间

## 2. ModelCheckpoint（模型检查点）
在训练过程中，把"验证集表现最好"的模型参数保存下来。
- 即使后面训练变差，仍能拿到最好的模型
- 方便后续加载和部署

### 两者一起使用的意义
- `EarlyStopping` 负责"及时停下来"
- `ModelCheckpoint` 负责"把最好的模型保存下来"

这是实际深度学习项目中非常常见的搭配。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import imdb
from tensorflow.keras import models
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import load_model

## 1. 加载 IMDB 数据集

- 标签 0：负面评论
- 标签 1：正面评论

只保留训练集中最常见的前 10000 个单词。

In [ ]:
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

print("训练集样本数：", len(train_data))
print("测试集样本数：", len(test_data))

## 2. 数据向量化

将每条评论转换成长度为 10000 的向量：
- 某个单词出现过，对应位置记为 1
- 没有出现，则记为 0

In [ ]:
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.
    return results

In [ ]:
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

y_train = np.asarray(train_labels).astype("float32")
y_test = np.asarray(test_labels).astype("float32")

print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)

## 3. 划分验证集

- 前 10000 条作为验证集
- 后面的样本作为真正的训练集

In [ ]:
x_val = x_train[:10000]
partial_x_train = x_train[10000:]

y_val = y_train[:10000]
partial_y_train = y_train[10000:]

print("验证集形状：", x_val.shape, y_val.shape)
print("训练集形状：", partial_x_train.shape, partial_y_train.shape)

## 4. 构建二分类模型

- 第一隐藏层：16个神经元，ReLU
- 第二隐藏层：16个神经元，ReLU
- 输出层：1个神经元，Sigmoid

In [ ]:
model = models.Sequential()
model.add(layers.Dense(16, activation="relu", input_shape=(10000,)))
model.add(layers.Dense(16, activation="relu"))
model.add(layers.Dense(1, activation="sigmoid"))

model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. 定义 EarlyStopping 和 ModelCheckpoint

### EarlyStopping 参数说明
- `monitor='val_loss'`：监控验证损失
- `patience=2`：连续2轮没有改善则停止
- `restore_best_weights=True`：自动恢复最佳参数

### ModelCheckpoint 参数说明
- `filepath='best_imdb_model.keras'`：保存最佳模型的文件名
- `monitor='val_loss'`：监控验证损失
- `save_best_only=True`：只保存验证集表现最好的模型
- `mode='min'`：损失越小越好

In [ ]:
# 早停机制：当验证损失不再下降时自动停止
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

# 模型保存：保存验证集表现最好的模型
checkpoint = ModelCheckpoint(
    filepath="best_imdb_model.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

## 6. 开始训练模型

- 最大训练 20 轮
- 如果验证损失不再改善，`EarlyStopping` 会让训练提前结束
- 每当验证损失刷新最好成绩时，`ModelCheckpoint` 自动保存当前最佳模型

In [ ]:
history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    callbacks=[early_stop, checkpoint]
)

## 7. 查看训练历史记录

In [ ]:
history_dict = history.history
print(history_dict.keys())

In [ ]:
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, loss_values, "bo", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()
plt.grid(True)
plt.show()

### 损失曲线解读

- 训练损失不断下降
- 验证损失先下降后可能上升（过拟合）
- 早停机制会在验证损失不再改善时自动停止

In [ ]:
acc_values = history_dict["accuracy"]
val_acc_values = history_dict["val_accuracy"]

plt.figure(figsize=(8, 5))
plt.plot(epochs, acc_values, "bo", label="Training accuracy")
plt.plot(epochs, val_acc_values, "b", label="Validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 8. 加载训练过程中保存的最佳模型

即使训练结束、Notebook关闭、程序重启，也可以重新使用保存的最佳模型。

In [ ]:
best_model = load_model("best_imdb_model.keras")
print("最佳模型加载成功。")
best_model.summary()

## 9. 在测试集上评估最佳模型

In [ ]:
results = best_model.evaluate(x_test, y_test)
print("测试损失：", results[0])
print("测试准确率：", results[1])

## 10. 使用最佳模型进行预测

sigmoid 输出 0~1 之间的概率值：
- 接近 1：正面评论
- 接近 0：负面评论

In [ ]:
predictions = best_model.predict(x_test)

print("前10条预测概率：")
print(predictions[:10])

In [ ]:
predicted_labels = (predictions >= 0.5).astype("int32")
print("前20条预测类别：")
print(predicted_labels[:20].reshape(-1))

## 11. 实验总结

本实验在 IMDB 影评二分类任务中，同时加入了：

| 机制 | 作用 |
|------|------|
| `EarlyStopping` | 防止过拟合、节省训练时间、自动恢复最佳权重 |
| `ModelCheckpoint` | 保存验证集表现最好的模型、方便后续加载部署 |

### 主要流程
1. 加载 IMDB 数据集
2. 将文本整数序列向量化
3. 划分训练集和验证集
4. 构建二分类神经网络
5. 使用 `EarlyStopping` 自动停止训练
6. 使用 `ModelCheckpoint` 保存验证集最佳模型
7. 重新加载最佳模型
8. 在测试集上评估最佳模型效果

### 实际项目中的意义
在实际深度学习训练中，这两个回调函数是常见的搭配：
- 一个负责"及时停"
- 一个负责"保存最好结果"

这样训练过程更稳、更规范。